# Review and clean duplicate synthesis experiments

This is database maintenance, not synthesis or benchmarking. **Default: preview only.** Stop all synthesis/evaluation jobs before applying changes. No migrations or campaigns are run.

Safe mode preserves all valid completed runs and interrupted sessions with recorded progress. Optional one-per-condition mode removes other completed repeats too: this discards experimental history and synthesis-repeat reliability evidence.

Groups use exact full model name, function, dimension, instance, explicit/implicit mode, noise level/model, strategy, budget and maximum iterations. Different scientific protocols are never merged. A completed finite champion counts as valid; this does not mean it solves the independent benchmark.


In [11]:
from dataclasses import asdict
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sqlalchemy.engine import make_url

cwd = Path.cwd().resolve()
project_root = next(
    root for root in (cwd, *cwd.parents)
    if (root / "pyproject.toml").is_file() and (root / "src/evolution").is_dir()
)
src = str(project_root / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from shared.config import DATABASE_URL, DATA_DIR
from shared.infra.database import Database
from evolution.application.maintenance.duplicates import SynthesisDuplicateMaintenance
from evolution.infra.storage.synthesis.maintenance import SQLiteDuplicateStorage

url = make_url(DATABASE_URL)
if not url.drivername.startswith("sqlite") or not url.database or url.database == ":memory:":
    raise ValueError("Use an existing file-backed SQLite database.")
database_path = Path(url.database)
if not database_path.is_file():
    raise FileNotFoundError(f"Database not found; refusing to create an empty database: {database_path}")

database = Database(DATABASE_URL)
maintenance = SynthesisDuplicateMaintenance(SQLiteDuplicateStorage(database))
print(f"Database: {database_path}")
print(f"Artifact directory: {DATA_DIR}")
pd.set_option("display.max_colwidth", 100)


Database: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/db.sqlite3
Artifact directory: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data


## 1. Choose the scope

Leave `MODELS = []` to discover all database models, or enter exact model names. Safe mode removes redundant failed/completed-without-champion records and empty queued duplicates while retaining a representative for every condition. Interrupted runs with progress remain protected.

Only set `COLLAPSE_VALID_RUNS = True` if you explicitly want one retained experiment per condition. The scientific ranking is final error, then evaluations (NULL first), with stable iteration-ID order for ties. Missing code is shown for investigation; it is not a new ranking criterion. If no valid completed result exists, retain the most-progressed pending run; otherwise retain the latest failed record for retry.


In [12]:
MODELS = []  # Example: ["qwen2.5-coder-14b-instruct-q4_k_m.gguf"]
COLLAPSE_VALID_RUNS = True
QUARANTINE_FILES = True  # Move exact old code/checkpoint folders into the backup.


In [13]:
plan = maintenance.preview(models=MODELS, collapse_valid_runs=COLLAPSE_VALID_RUNS)
report = pd.DataFrame([
    {
        "experiment_id": row.experiment_id,
        **asdict(row.condition),
        "status": row.status,
        "iterations": row.iterations,
        "champion_error": row.champion_error,
        "evaluations": row.evaluations,
        "code_available": row.code_available,
        "valid_completed": row.valid_completed,
        "action": row.action,
        "reason": row.reason,
    }
    for row in plan.rows
])
if report.empty:
    print("No duplicate conditions in this scope.")
else:
    display(report.groupby(["model", "action"]).size().rename("records").to_frame())
    display(report.sort_values(["model", "problem_id", "dimension", "mode", "noise", "strategy", "experiment_id"]))
print("Exact experiment IDs proposed for removal:", list(plan.delete_ids))
print("Required confirmation:", plan.confirmation)
print("Preview is read-only. No records or artifacts have been removed.")


records
model                                       action         
DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf delete       68
                                            keep         59
qwen2.5-coder-14b-instruct-q4_k_m.gguf      delete     2685
                                            keep        640
qwen2.5-coder-32b-instruct-q4_k_m.gguf      delete       25
                                            keep         23

,experiment_id,model,problem_id,dimension,instance,mode,noise,noise_model,strategy,budget,max_iterations,status,iterations,champion_error,evaluations,code_available,valid_completed,action,reason
3373,2095,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,1,3,1,explicit,0.00,none,baseline,1000000,10,completed,10,NaN,NaN,False,False,delete,Redundant record; representative #2703 retained
3374,2703,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,1,3,1,explicit,0.00,none,baseline,1000000,10,completed,10,8.288622e-08,20001.0,True,True,keep,Retained representative
3375,2114,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,1,3,1,explicit,0.10,heteroscedastic,guided,1000000,10,completed,10,NaN,NaN,False,False,delete,Redundant record; representative #2704 retained
3376,2704,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,1,3,1,explicit,0.10,heteroscedastic,guided,1000000,10,completed,10,0.000000e+00,1000020.0,True,True,keep,Retained representative
3377,2119,DeepSeek-Coder-V2-Lite-Instruct-Q4_K_M.gguf,1,3,1,explicit,0.20,heteroscedastic,baseline,1000000,10,completed,10,NaN,NaN,False,False,delete,Redundant record; representative #2705 retained
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3368,1418,qwen2.5-coder-32b-instruct-q4_k_m.gguf,15,10,1,explicit,0.05,heteroscedastic,baseline,1000000,10,completed,10,5.060285e+00,1000050.0,True,True,keep,Retained representative
3369,1264,qwen2.5-coder-32b-instruct-q4_k_m.gguf,15,10,1,explicit,0.20,heteroscedastic,baseline,1000000,10,completed,10,NaN,NaN,False,False,delete,Redundant record; representative #1419 retained
3370,1419,qwen2.5-coder-32b-instruct-q4_k_m.gguf,15,10,1,explicit,0.20,heteroscedastic,baseline,1000000,10,completed,10,2.108428e+01,1000000.0,True,True,keep,Retained representative
3371,1290,qwen2.5-coder-32b-instruct-q4_k_m.gguf,21,2,1,explicit,0.10,heteroscedastic,vectorization,1000000,10,completed,10,NaN,NaN,False,False,delete,Redundant record; representative #1420 retained


Exact experiment IDs proposed for removal: [2791, 2792, 2793, 2794, 2795, 2796, 2797, 2798, 2799, 2800, 2801, 2802, 2803, 2804, 2805, 2806, 5, 2807, 2809, 2810, 2811, 2812, 2813, 2814, 2815, 2816, 2817, 2818, 2819, 2820, 2821, 2822, 2823, 2824, 2825, 2826, 10, 2827, 2828, 2829, 2830, 2831, 2832, 2833, 2834, 2835, 2836, 2837, 2838, 13, 2839, 2840, 2841, 2842, 2843, 2844, 2845, 2846, 2847, 2848, 2849, 2850, 2851, 2852, 2853, 2854, 2855, 2856, 2857, 2858, 2859, 2860, 2861, 2862, 19, 643, 2863, 2864, 2865, 2866, 2867, 2868, 2869, 2870, 2871, 2872, 2873, 2874, 2875, 2876, 2877, 2878, 2879, 2880, 2881, 2882, 24, 644, 2883, 2884, 2885, 2886, 2887, 2888, 2889, 2890, 2891, 2892, 2893, 2894, 27, 2895, 2896, 2897, 2898, 2899, 2900, 2901, 2902, 29, 2903, 2904, 2905, 2906, 2907, 2908, 2909, 2910, 2911, 2912, 2913, 2914, 32, 2915, 2916, 2917, 2918, 650, 658, 2919, 2920, 2921, 2922, 651, 2923, 2924, 2925, 2926, 652, 2927, 2928, 2929, 2930, 2931, 2932, 2933, 2934, 2935, 2936, 2937, 2938, 2939, 2940, 2

## 2. Apply only after reviewing the exact IDs

This cell defaults to doing nothing. To apply, stop every synthesis/evaluation process, set `JOBS_STOPPED = True`, type the exact confirmation printed above, and set `APPLY_CLEANUP = True`.

A writer lock prevents concurrent database writes during cleanup. If database contents or candidate availability changed since preview, cleanup refuses to run: rerun the preview and review it again. The backup includes the full SQLite database and the exact plan. When enabled, file cleanup quarantines only matching `data/code/experiment_ID` and `data/evolution_state/.../experiment_ID` directories. Symbolic links and code referenced by retained experiments are rejected.

Benchmark traces, champion exports and figures are not deleted. Records outside your duplicate groups/model scope remain untouched. The operation checks that champion-covered conditions remain covered before committing.


In [14]:
APPLY_CLEANUP = False
JOBS_STOPPED = False
CONFIRMATION = ""  # Type the exact printed phrase, e.g. "DELETE 12 EXPERIMENTS".

cleanup_result = None
if not APPLY_CLEANUP:
    print("Preview only: cleanup has not been applied.")
elif not plan.delete_ids:
    print("Nothing to remove.")
else:
    cleanup_result = maintenance.apply(
        plan,
        confirmation=CONFIRMATION,
        jobs_stopped=JOBS_STOPPED,
        quarantine_files=QUARANTINE_FILES,
    )
    print("Removed experiment records:", list(cleanup_result.deleted_ids))
    print("Recoverable SQLite backup:", cleanup_result.backup_directory / "database.sqlite3")
    print("Plan and file mapping:", cleanup_result.backup_directory / "manifest.json")
    print("Quarantined directories:", len(cleanup_result.quarantined_paths))
    after = maintenance.preview(models=MODELS, collapse_valid_runs=COLLAPSE_VALID_RUNS)
    print("Remaining proposed duplicate IDs:", list(after.delete_ids))


Preview only: cleanup has not been applied.


## 3. Verify downstream outputs and recovery

Rerun the synthesis audit, then reload champion selection/readiness in Notebook 03. Cached exports referencing removed IDs may be stale; rerun only affected benchmark conditions identified by readiness/provenance. Do not assume old traces become valid merely because a directory still exists. No figures or evaluations are regenerated by this notebook.

The full database backup is at the printed path; quarantined files retain their original paths relative to `DATA_DIR` under `files/`. To restore, stop all jobs and close database connections first. Restore the **whole database** (not selected rows with conflicting IDs), handle its SQLite WAL/SHM sidecars, and restore files from `files/` to their original paths. Do not overwrite a live database. Recovery is intentionally manual; the notebook never silently replaces the live database.


In [15]:
database.dispose()
